# Risk Management — Case 2026-I
Middle-office risk system: volatilidad condicional, optimizacion dinamica y riesgo de cola.
Datos: Yahoo Finance (yfinance), enero 2018 – diciembre 2025.

In [ ]:
# Setup — librerías, parámetros y descarga de datos (Yahoo Finance)

!pip install -q yfinance arch cvxpy
import yfinance as yf

def download_prices():
    raw = yf.download(TICKERS, start=DATA_START, end=DATA_END,
                      auto_adjust=True, progress=False)["Close"]
    return raw[TICKERS].ffill(limit=3).dropna()

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from scipy.stats import skew, kurtosis
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
from arch import arch_model
import cvxpy as cp

TICKERS = ["SPY","EEM","EWJ","TIP","SHY","TLT","HYG","LQD","DBA","GLD","USO","VIXY","VNQ","QQQ","SOXX"]
ASSETS = {"SPY":"S&P 500 ETF","EEM":"Emerging Markets ETF","EWJ":"Japan ETF","TIP":"TIPS ETF",
          "SHY":"Short-Term UST","TLT":"Long-Term UST","HYG":"High Yield Corp.","LQD":"Invest. Grade Corp.",
          "DBA":"Agriculture","GLD":"Gold ETF","USO":"Oil ETF","VIXY":"VIX Futures","VNQ":"Real Estate ETF",
          "QQQ":"Nasdaq 100 ETF","SOXX":"Semiconductors ETF"}
DATA_START, DATA_END = "2018-01-01", "2025-12-31"
INSAMPLE_END, OOS_START = "2023-12-31", "2024-01-01"
ARCH_LM_LAGS, LJUNGBOX_LAGS, ARCH_Q = 10, [10, 20], 5
TRADING_DAYS, ALPHA = 252, 0.975
KEY_PAIRS = [("SPY","TLT"),("SPY","HYG"),("SPY","GLD"),("SPY","VIXY"),("QQQ","SOXX"),("EEM","SPY")]
plt.rcParams.update({"figure.facecolor":"white","axes.grid":True,"grid.alpha":0.3})

def to_log_returns(prices, pct=True):
    lr = np.log(prices / prices.shift(1)).dropna()
    return lr * 100 if pct else lr

prices = download_prices()
returns = to_log_returns(prices, pct=True)
in_sample = returns.loc[:INSAMPLE_END]
oos = returns.loc[OOS_START:]
print("Precios:", prices.shape, "| Retornos:", returns.shape,
      "| In-sample:", in_sample.shape, "| OOS:", oos.shape)
print("Periodo:", returns.index[0].date(), "a", returns.index[-1].date())

In [ ]:
# Pregunta 1 — Exploratory Data Analysis (2 pts)
def descriptive_stats(r):
    rows = []
    for c in r.columns:
        s = r[c].dropna()
        jb, jbp = stats.jarque_bera(s)
        rows.append({"Activo":c,"Media (%)":s.mean(),"Std diaria (%)":s.std(),
                     "Std anual (%)":s.std()*np.sqrt(TRADING_DAYS),"Skewness":stats.skew(s),
                     "Exceso Kurtosis":stats.kurtosis(s),"Min (%)":s.min(),"Max (%)":s.max(),
                     "JB p-valor":jbp,"Rechaza Normal (5%)":jbp<0.05})
    return pd.DataFrame(rows).set_index("Activo").round(4)

def ljungbox_summary(r, lags=(10,20)):
    rows = []
    for c in r.columns:
        s = r[c].dropna()
        lb1 = acorr_ljungbox(s, lags=list(lags), return_df=True)
        lb2 = acorr_ljungbox(s**2, lags=list(lags), return_df=True)
        row = {"Activo":c}
        for lag in lags:
            row[f"LB(r) p@{lag}"] = lb1.loc[lag,"lb_pvalue"]
            row[f"LB(r2) p@{lag}"] = lb2.loc[lag,"lb_pvalue"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("Activo").round(4)

def engle_lm_test(r, nlags=ARCH_LM_LAGS):
    rows = []
    for c in r.columns:
        s = r[c].dropna()
        lm, lmp, _, _ = het_arch(s, nlags=nlags)
        rows.append({"Activo":c,"LM stat":lm,"LM p-valor":lmp,"Rechaza H0 (ARCH)":lmp<0.05})
    return pd.DataFrame(rows).set_index("Activo").round(4)

def plot_prices_returns(prices, returns):
    norm = 100*prices/prices.iloc[0]
    n = len(TICKERS)
    fig, axes = plt.subplots(n, 2, figsize=(13, 2.0*n))
    fig.suptitle("Pregunta 1 - Precios normalizados (base 100) y log-retornos", fontweight="bold", y=1.003)
    for i, t in enumerate(TICKERS):
        ap, ar = axes[i,0], axes[i,1]
        ap.plot(norm.index, norm[t], lw=0.8, color="#2c3e50")
        ap.axvline(pd.Timestamp(INSAMPLE_END), color="red", ls="--", lw=0.8)
        ap.set_title(f"{t} - {ASSETS[t]}", fontsize=8, loc="left")
        ar.plot(returns.index, returns[t], lw=0.4, color="#2980b9", alpha=0.8)
        ar.axvline(pd.Timestamp(INSAMPLE_END), color="red", ls="--", lw=0.8)
        ar.set_title(f"{t} - retornos (%)", fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

def plot_acf_grid(returns, subset=("SPY","QQQ","SOXX","TLT","HYG","GLD"), lags=30):
    fig, axes = plt.subplots(len(subset), 2, figsize=(11, 2.2*len(subset)))
    fig.suptitle("Pregunta 1 - ACF(r) vs ACF(r^2): volatility clustering", fontweight="bold", y=1.005)
    for i, t in enumerate(subset):
        s = returns[t].dropna()
        plot_acf(s, lags=lags, ax=axes[i,0], color="#2980b9", title=f"{t}: ACF(r)")
        plot_acf(s**2, lags=lags, ax=axes[i,1], color="#e74c3c", title=f"{t}: ACF(r^2)")
    plt.tight_layout(); plt.show()

def plot_histograms(returns, subset=("SPY","QQQ","SOXX","TLT","HYG","GLD")):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    fig.suptitle("Pregunta 1 - Distribucion empirica vs Normal (fat tails)", fontweight="bold")
    for ax, t in zip(axes.flat, subset):
        s = returns[t].dropna()
        x = np.linspace(s.min(), s.max(), 300)
        ax.hist(s, bins=80, density=True, alpha=0.6, color="#3498db", label="Empirico")
        ax.plot(x, stats.norm.pdf(x, s.mean(), s.std()), color="black", lw=1.5, label="Normal")
        ax.set_title(f"{t} (exc.kurt={stats.kurtosis(s):.2f})", fontsize=9); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

desc = descriptive_stats(returns)
lm = engle_lm_test(returns)
lb = ljungbox_summary(returns)
print("ESTADISTICOS DESCRIPTIVOS"); print(desc.to_string())
print("\nTEST LM DE ENGLE (ARCH effect)"); print(lm.to_string())
print("\nLJUNG-BOX en r y r^2"); print(lb.to_string())
plot_prices_returns(prices, returns)
plot_acf_grid(returns)
plot_histograms(returns)

In [ ]:
# Pregunta 2 — Volatilidad condicional: ARCH/GARCH/GJR/EGARCH (5 pts)
def sign_bias_test(res):
    z2 = res.std_resid**2
    eps = res.resid
    Sm = (eps.shift(1) < 0).astype(int)
    Sp = 1 - Sm
    df = pd.DataFrame({"z2":z2,"Sm":Sm,"Sm_eps":Sm*eps.shift(1),"Sp_eps":Sp*eps.shift(1)}).dropna()
    X = sm.add_constant(df[["Sm","Sm_eps","Sp_eps"]])
    ols = sm.OLS(df["z2"], X).fit(cov_type="HC3")
    ft = ols.f_test("(Sm = 0), (Sm_eps = 0), (Sp_eps = 0)")
    fp = float(ft.pvalue)
    fv = float(ft.fvalue[0][0]) if hasattr(ft.fvalue, "__len__") else float(ft.fvalue)
    return {"F_stat":fv,"F_pvalue":fp,"asimetria":fp<0.05}

def fit_all_specs(r):
    r = r.dropna()
    out = {}
    out["ARCH(q)"] = arch_model(r, mean="Constant", vol="ARCH", p=ARCH_Q, dist="t").fit(disp="off")
    out["GARCH(1,1)-N"] = arch_model(r, mean="Constant", vol="Garch", p=1, q=1, dist="normal").fit(disp="off")
    out["GARCH(1,1)-t"] = arch_model(r, mean="Constant", vol="Garch", p=1, q=1, dist="t").fit(disp="off")
    out["GJR(1,1)-t"] = arch_model(r, mean="Constant", vol="Garch", p=1, o=1, q=1, dist="t").fit(disp="off")
    out["EGARCH(1,1)-t"] = arch_model(r, mean="Constant", vol="EGARCH", p=1, o=1, q=1, dist="t").fit(disp="off")
    return out

def select_best_model(models):
    return min({n:m.bic for n,m in models.items()}, key=lambda k: models[k].bic)

def diagnose_residuals(res, name):
    z = res.std_resid.dropna()
    lb = acorr_ljungbox(z**2, lags=LJUNGBOX_LAGS, return_df=True)
    d = {"modelo":name,"z_mean":z.mean(),"z_std":z.std(),"z_skew":skew(z),"z_exkurt":kurtosis(z),
         "LB_z2_p10":lb.loc[10,"lb_pvalue"],"LB_z2_p20":lb.loc[20,"lb_pvalue"],
         "bien_especificado":(lb.loc[10,"lb_pvalue"]>0.05) and (lb.loc[20,"lb_pvalue"]>0.05)}
    sb = sign_bias_test(res)
    d["sign_bias_p"] = sb["F_pvalue"]; d["asimetria_residual"] = sb["asimetria"]
    return d

def run_asset_pipeline(ticker, r, verbose=True):
    models = fit_all_specs(r)
    base_sb = sign_bias_test(models["GARCH(1,1)-t"])
    best = select_best_model(models)
    diag = diagnose_residuals(models[best], best)
    comp = pd.DataFrame({n:{"AIC":m.aic,"BIC":m.bic,"LogLik":m.loglikelihood}
                         for n,m in models.items()}).T.round(2)
    if verbose:
        print(f"\n--- {ticker} ({ASSETS[ticker]}) ---"); print(comp)
        print(f"  Sign Bias (GARCH-t): p={base_sb['F_pvalue']:.4f} -> "
              f"{'ASIMETRIA -> GJR/EGARCH' if base_sb['asimetria'] else 'simetrico OK'}")
        print(f"  Modelo elegido (BIC): {best} | bien especificado: {diag['bien_especificado']}")
    return {"ticker":ticker,"models":models,"comparison":comp,"baseline_sign_bias":base_sb,
            "best_model_name":best,"best_model":models[best],"diagnostics":diag}

def plot_conditional_vol_panel(res_by_asset):
    n = len(res_by_asset)
    fig, axes = plt.subplots(n, 1, figsize=(11, 1.8*n))
    fig.suptitle("Pregunta 2 - Volatilidad condicional anualizada (modelo ganador)", fontweight="bold", y=1.003)
    for ax, (t, d) in zip(axes, res_by_asset.items()):
        v = d["best_model"].conditional_volatility*np.sqrt(TRADING_DAYS)
        ax.plot(v.index, v.values, lw=0.7, color="#8e44ad")
        ax.set_title(f"{t} - {d['best_model_name']}", fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

print("Estimando 5 especificaciones x 15 activos (puede tardar varios minutos)...")
fitted = {}
rows = []
for t in TICKERS:
    res = run_asset_pipeline(t, in_sample[t], verbose=True)
    fitted[t] = res
    rows.append({"Activo":t,"Modelo ganador (BIC)":res["best_model_name"],
                 "Asimetria baseline":res["baseline_sign_bias"]["asimetria"],
                 "Bien especificado":res["diagnostics"]["bien_especificado"],
                 "Exc.kurt z":round(res["diagnostics"]["z_exkurt"],3)})
summary = pd.DataFrame(rows).set_index("Activo")
print("\nRESUMEN - Modelo ganador por activo"); print(summary.to_string())
plot_conditional_vol_panel(fitted)

In [ ]:
# Pregunta 2 (cont.) — DCC (Engle 2002): matriz de covarianza condicional
def build_standardized_residuals(fitted_models):
    z = pd.DataFrame({t: fitted_models[t]["best_model"].std_resid for t in fitted_models})
    return z.dropna()

def dcc_loglik(params, Z, Qbar):
    a, b = params
    T, N = Z.shape
    if a < 0 or b < 0 or a + b >= 0.999:
        return 1e10
    Q = Qbar.copy(); ll = 0.0
    for t in range(1, T):
        zl = Z[t-1]; Q = (1-a-b)*Qbar + a*np.outer(zl, zl) + b*Q
        d = np.sqrt(np.diag(Q)); R = Q/np.outer(d, d)
        try:
            Ri = np.linalg.inv(R); sign, logdet = np.linalg.slogdet(R)
        except np.linalg.LinAlgError:
            return 1e10
        if sign <= 0:
            return 1e10
        zt = Z[t]; ll += -0.5*(logdet + zt@Ri@zt - zt@zt)
    return -ll

def fit_dcc(z_df, x0=(0.02, 0.95)):
    Z = z_df.values; Qbar = np.corrcoef(Z, rowvar=False)
    res = minimize(dcc_loglik, x0=x0, args=(Z, Qbar), method="Nelder-Mead",
                   options={"xatol":1e-5,"fatol":1e-5,"maxiter":500})
    a, b = res.x
    T, N = Z.shape; Q = Qbar.copy()
    R_series = np.zeros((T, N, N)); R_series[0] = Qbar
    for t in range(1, T):
        zl = Z[t-1]; Q = (1-a-b)*Qbar + a*np.outer(zl, zl) + b*Q
        d = np.sqrt(np.diag(Q)); R_series[t] = Q/np.outer(d, d)
    return {"a":a,"b":b,"persistence":a+b,"Qbar":Qbar,"R_t":R_series,
            "loglik":-res.fun,"tickers":list(z_df.columns),"dates":z_df.index}

def build_conditional_covariance(dcc, cond_vols):
    tk = dcc["tickers"]; D = cond_vols[tk].reindex(dcc["dates"]).values
    R = dcc["R_t"]; T, N = D.shape
    S = np.zeros((T, N, N))
    for t in range(T):
        Dt = np.diag(D[t]); S[t] = Dt@R[t]@Dt
    return S

def build_ccc_covariance(z_df, cond_vols):
    tk = list(z_df.columns); Rc = np.corrcoef(z_df.values, rowvar=False)
    D = cond_vols[tk].reindex(z_df.index).values; T, N = D.shape
    S = np.zeros((T, N, N))
    for t in range(T):
        Dt = np.diag(D[t]); S[t] = Dt@Rc@Dt
    return S

def plot_dynamic_correlations(dcc, ccc_R):
    tk = dcc["tickers"]; idx = {t:tk.index(t) for t in tk}; dates = dcc["dates"]
    fig, axes = plt.subplots(3, 2, figsize=(13, 10))
    fig.suptitle("Pregunta 2 (cont.) - Correlacion condicional DCC vs CCC", fontweight="bold", y=1.005)
    for ax, (a, b) in zip(axes.flat, KEY_PAIRS):
        i, j = idx[a], idx[b]
        ax.plot(dates, dcc["R_t"][:, i, j], color="#c0392b", lw=0.9, label="DCC")
        ax.axhline(ccc_R[i, j], color="#2c3e50", ls="--", lw=1.0, label="CCC")
        ax.axhline(0, color="gray", lw=0.4)
        ax.set_title(f"Corr({a},{b})", fontsize=10); ax.legend(fontsize=7, loc="upper left")
    plt.tight_layout(); plt.show()

z_df = build_standardized_residuals(fitted)
cond_vols = pd.DataFrame({t: fitted[t]["best_model"].conditional_volatility for t in TICKERS}).reindex(z_df.index)
print("Estimando DCC(1,1) por maxima verosimilitud sobre z_t...")
dcc_res = fit_dcc(z_df)
print(f"  a_hat = {dcc_res['a']:.4f} | b_hat = {dcc_res['b']:.4f} | a+b = {dcc_res['persistence']:.4f}")
print(f"  Log-verosimilitud = {dcc_res['loglik']:.2f}")
Sigma_dcc = build_conditional_covariance(dcc_res, cond_vols)
Sigma_ccc = build_ccc_covariance(z_df, cond_vols)
ccc_R = np.corrcoef(z_df.values, rowvar=False)
print(f"  Sigma_t construida: forma {Sigma_dcc.shape}")
plot_dynamic_correlations(dcc_res, ccc_R)

In [ ]:
# Pregunta 3 — Mean-Variance y Risk Parity (5 pts)
REBAL_FREQ_DAYS, VOL_SHOCK_THRESHOLD = 21, 1.5

def solve_mean_variance(Sigma):
    n = Sigma.shape[0]; Sp = (Sigma+Sigma.T)/2 + 1e-8*np.eye(n)
    w = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, Sp)), [cp.sum(w)==1, w>=0])
    prob.solve(solver=cp.CLARABEL)
    if w.value is not None and prob.status == "optimal":
        return np.array(w.value).flatten()
    w0 = np.ones(n)/n
    res = minimize(lambda x: x@Sp@x, w0, method="SLSQP", bounds=[(0,1)]*n,
                   constraints=[{"type":"eq","fun":lambda x: np.sum(x)-1}])
    return res.x

def solve_risk_parity(Sigma):
    n = Sigma.shape[0]; Sp = (Sigma+Sigma.T)/2 + 1e-8*np.eye(n)
    w = cp.Variable(n, pos=True)
    obj = cp.Minimize(0.5*cp.quad_form(w, Sp) - (1.0/n)*cp.sum(cp.log(w)))
    prob = cp.Problem(obj); prob.solve(solver=cp.CLARABEL)
    if w.value is not None and prob.status == "optimal":
        wr = np.array(w.value).flatten(); return wr/wr.sum()
    def f(x):
        if np.any(x <= 0): return 1e10
        return 0.5*x@Sp@x - (1.0/n)*np.sum(np.log(x))
    res = minimize(f, np.ones(n)*0.5, method="Nelder-Mead",
                   options={"xatol":1e-8,"fatol":1e-10,"maxiter":5000})
    return res.x/res.x.sum()

def risk_contributions(w, Sigma):
    pv = w@Sigma@w; return (w*(Sigma@w))/pv

def run_portfolio_backtest(returns, Sigma_t, dates):
    rebal = dates[::REBAL_FREQ_DAYS]
    wmv_h, wrp_h, mv_r, rp_r = [], [], [], []
    wmv = wrp = None
    for i, dt in enumerate(dates):
        if dt in rebal or wmv is None:
            wmv = solve_mean_variance(Sigma_t[i]); wrp = solve_risk_parity(Sigma_t[i])
        wmv_h.append(wmv.copy()); wrp_h.append(wrp.copy())
        r = returns.loc[dt, TICKERS].values/100
        mv_r.append(float(wmv@r)); rp_r.append(float(wrp@r))
    return {"weights_mv":pd.DataFrame(wmv_h, index=dates, columns=TICKERS),
            "weights_rp":pd.DataFrame(wrp_h, index=dates, columns=TICKERS),
            "returns_mv":pd.Series(mv_r, index=dates, name="MeanVariance"),
            "returns_rp":pd.Series(rp_r, index=dates, name="RiskParity"),
            "rebal_dates":rebal}

def identify_event_rebalance_dates(pr):
    v5 = pr.rolling(5).std(); v60 = v5.rolling(60).mean()
    return pr.index[(v5 > VOL_SHOCK_THRESHOLD*v60).fillna(False)]

def plot_weight_evolution(w, title):
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.stackplot(w.index, w.T.values, labels=w.columns, alpha=0.85)
    ax.set_title(title, fontweight="bold"); ax.set_ylim(0, 1)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=7)
    plt.tight_layout(); plt.show()

print(f"Resolviendo MV y RP con rebalanceo cada {REBAL_FREQ_DAYS} dias habiles...")
bt = run_portfolio_backtest(in_sample.loc[z_df.index], Sigma_dcc, z_df.index)
rc_last = risk_contributions(bt["weights_rp"].iloc[-1].values, Sigma_dcc[-1])
print("\nVerificacion Risk Parity (contribuciones de riesgo ~ 1/15 = 0.067):")
print(pd.Series(rc_last, index=TICKERS).round(4).to_string())
ev_mv = identify_event_rebalance_dates(bt["returns_mv"])
ev_rp = identify_event_rebalance_dates(bt["returns_rp"])
print(f"\nRegla de rebalanceo: (a) calendario cada {REBAL_FREQ_DAYS} dias + "
      f"(b) evento si vol_5d > {VOL_SHOCK_THRESHOLD}x vol_60d")
print(f"Eventos detectados - MV: {len(ev_mv)} | RP: {len(ev_rp)}")
print("\nPesos promedio en la muestra:")
print(pd.DataFrame({"MV":bt['weights_mv'].mean(),"RP":bt['weights_rp'].mean()}).round(4).to_string())
plot_weight_evolution(bt["weights_mv"], "Pregunta 3 - Pesos Mean-Variance en el tiempo")
plot_weight_evolution(bt["weights_rp"], "Pregunta 3 - Pesos Risk Parity en el tiempo")

In [ ]:
# Pregunta 4 — VaR, ES y EVT al 97.5% (4 pts)
def parametric_var_es(mu, sigma, alpha=ALPHA, dist="t", nu=None):
    p = 1 - alpha
    if dist == "normal":
        q = stats.norm.ppf(p)
        return {"VaR":-(mu+sigma*q),"ES":-(mu - sigma*stats.norm.pdf(q)/p)}
    if nu is None or nu <= 2:
        raise ValueError("nu>2 requerido")
    sc = np.sqrt((nu-2)/nu); qr = stats.t.ppf(p, nu); q = qr*sc
    es_raw = (stats.t.pdf(qr, nu)/p)*((nu+qr**2)/(nu-1))
    return {"VaR":-(mu+sigma*q),"ES":-(mu - sigma*sc*es_raw)}

def historical_var_es(r, alpha=ALPHA):
    x = r.dropna().values; thr = np.percentile(x, (1-alpha)*100)
    tail = x[x <= thr]
    return {"VaR":-thr,"ES":-tail.mean() if len(tail) else np.nan}

def fit_gpd_pot(losses, tq=0.90):
    u = np.percentile(losses, tq*100); exc = losses[losses > u] - u; ne = len(exc)
    def nll(p):
        xi, beta = p
        if beta <= 0: return 1e10
        if abs(xi) > 1e-8:
            z = 1 + xi*exc/beta
            if np.any(z <= 0): return 1e10
            return n_exc_term(ne, beta) + (1+1/xi)*np.sum(np.log(z))
        return n_exc_term(ne, beta) + np.sum(exc)/beta
    def n_exc_term(ne, beta): return ne*np.log(beta)
    res = minimize(nll, x0=[0.1, exc.std()], method="Nelder-Mead",
                   options={"xatol":1e-6,"fatol":1e-6})
    xi, beta = res.x
    return {"u":u,"xi":xi,"beta":beta,"n_exceedances":ne,"n_total":len(losses)}

def evt_var_es(losses, alpha=ALPHA, tq=0.90):
    g = fit_gpd_pot(losses, tq); u, xi, beta = g["u"], g["xi"], g["beta"]
    n, Nu = g["n_total"], g["n_exceedances"]; p = 1 - alpha
    if Nu == 0: return {"VaR":np.nan,"ES":np.nan, **g}
    if abs(xi) > 1e-6:
        var = u + (beta/xi)*(((n/Nu)*p)**(-xi) - 1)
        es = var/(1-xi) + (beta - xi*u)/(1-xi)
    else:
        var = u - beta*np.log((n/Nu)*p); es = var + beta
    return {"VaR":var,"ES":es, **g}

def evt_var_es_correct(z, mu, sigma, alpha=ALPHA):
    ez = evt_var_es(-z.dropna().values, alpha)
    if np.isnan(ez["VaR"]):
        return {"VaR":np.nan,"ES":np.nan,"xi":ez["xi"],"beta":ez["beta"]}
    return {"VaR":-mu+sigma*ez["VaR"],"ES":-mu+sigma*ez["ES"],"xi":ez["xi"],
            "beta":ez["beta"],"u_z":ez["u"],"n_exceedances":ez["n_exceedances"]}

def risk_table_for_asset(t, r, fm, alpha=ALPHA):
    best = fm["best_model"]; s = float(best.conditional_volatility.iloc[-1]); mu = 0.0
    nu = float(best.params["nu"]) if "nu" in best.params.index else float(stats.t.fit(best.std_resid.dropna())[0])
    pn = parametric_var_es(mu, s, alpha, "normal")
    pt = parametric_var_es(mu, s, alpha, "t", nu)
    h = historical_var_es(r, alpha); e = evt_var_es_correct(best.std_resid, mu, s, alpha)
    return {"Activo":t,"sigma_t (%)":s,"nu":nu,"VaR N (%)":pn["VaR"],"ES N (%)":pn["ES"],
            "VaR t (%)":pt["VaR"],"ES t (%)":pt["ES"],"VaR Hist (%)":h["VaR"],"ES Hist (%)":h["ES"],
            "VaR EVT (%)":e["VaR"],"ES EVT (%)":e["ES"],"xi GPD":e["xi"]}

def plot_var_comparison(rdf):
    cols = ["VaR N (%)","VaR t (%)","VaR Hist (%)","VaR EVT (%)"]
    fig, ax = plt.subplots(figsize=(13, 5))
    rdf.set_index("Activo")[cols].plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"Pregunta 4 - VaR al {ALPHA*100:g}% por familia de modelo", fontweight="bold")
    ax.set_ylabel("VaR diario (%)"); plt.tight_layout(); plt.show()

print(f"Calculando VaR/ES/EVT al {ALPHA*100:g}% para los 15 activos...")
risk_df = pd.DataFrame([risk_table_for_asset(t, in_sample[t], fitted[t]) for t in TICKERS]).round(4)
print(risk_df.to_string(index=False))
plot_var_comparison(risk_df)

def plot_evt_diagnostics(ticker, z):
    losses = -z.dropna().values
    qs = np.linspace(0.70, 0.975, 35)
    us = np.percentile(losses, qs*100)
    me = [(losses[losses>u]-u).mean() if (losses>u).sum()>=5 else np.nan for u in us]
    xis = [fit_gpd_pot(losses, q)["xi"] for q in qs]
    g = fit_gpd_pot(losses, 0.90); u0, xi0, b0 = g["u"], g["xi"], g["beta"]
    ex = np.sort(losses[losses>u0]-u0); n = len(ex); i = np.arange(1, n+1)
    theo = (b0/xi0)*((1-i/(n+1))**(-xi0)-1) if abs(xi0)>1e-6 else -b0*np.log(1-i/(n+1))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Pregunta 4 - Diagnostico EVT (POT/GPD) para {ticker}", fontweight="bold")
    ax[0].plot(us, me, "o-", color="#2980b9"); ax[0].axvline(u0, color="red", ls="--")
    ax[0].set_title("Mean Excess Plot"); ax[0].set_xlabel("umbral u"); ax[0].set_ylabel("e(u)")
    ax[1].plot(qs*100, xis, "o-", color="#8e44ad"); ax[1].axvline(90, color="red", ls="--")
    ax[1].set_title("Estabilidad de xi vs umbral"); ax[1].set_xlabel("percentil umbral (%)")
    ax[2].scatter(theo, ex, s=15, color="#27ae60"); lim = max(theo.max(), ex.max())
    ax[2].plot([0, lim], [0, lim], "r-"); ax[2].set_title("Q-Q: empirico vs GPD")
    plt.tight_layout(); plt.show()

for _t in ("SPY", "QQQ", "VIXY", "GLD"):
    plot_evt_diagnostics(_t, fitted[_t]["best_model"].std_resid)

In [ ]:
# Pregunta 5 — Backtesting OOS: Kupiec, Christoffersen y stress test (4 pts)
def rolling_portfolio_var(pr, sigma_p, alpha=ALPHA, span=60):
    nu = float(stats.t.fit(((pr - pr.mean())/pr.ewm(span=span).std()).dropna())[0])
    rows = []
    for t in sigma_p.index:
        s = sigma_p.loc[t]
        pn = parametric_var_es(0.0, s, alpha, "normal"); pt = parametric_var_es(0.0, s, alpha, "t", nu)
        rows.append({"date":t,"VaR_Normal":pn["VaR"],"ES_Normal":pn["ES"],"VaR_t":pt["VaR"],"ES_t":pt["ES"]})
    return pd.DataFrame(rows).set_index("date")

def add_evt_var(vdf, z_in, sigma_p, alpha=ALPHA):
    ez = evt_var_es_correct(z_in, 0.0, 1.0, alpha)
    vdf["VaR_EVT"] = sigma_p.values*ez["VaR"]; vdf["ES_EVT"] = sigma_p.values*ez["ES"]
    return vdf

def kupiec_test(exc, alpha=ALPHA):
    T = len(exc); N = int(exc.sum()); p = 1 - alpha; pi = N/T if T else 0.0
    if N == 0:
        lln = T*np.log(1-p); lla = 0.0
    elif N == T:
        lln = T*np.log(p); lla = T*np.log(pi)
    else:
        lln = (T-N)*np.log(1-p) + N*np.log(p); lla = (T-N)*np.log(1-pi) + N*np.log(pi)
    LR = -2*(lln - lla); pv = 1 - stats.chi2.cdf(LR, 1)
    return {"T":T,"N":N,"pi_hat":round(pi,4),"LR_uc":round(LR,3),"p_value":round(pv,4),"rechaza":pv<0.05}

def christoffersen_independence(exc):
    n00=n01=n10=n11=0
    for t in range(1, len(exc)):
        a, b = exc[t-1], exc[t]
        if a==0 and b==0: n00+=1
        elif a==0 and b==1: n01+=1
        elif a==1 and b==0: n10+=1
        else: n11+=1
    p01 = n01/(n00+n01) if (n00+n01) else 0.0
    p11 = n11/(n10+n11) if (n10+n11) else 0.0
    pc = (n01+n11)/(n00+n01+n10+n11) if (n00+n01+n10+n11) else 0.0
    sl = lambda x: np.log(x) if x>0 else 0.0
    llc = (n00+n10)*sl(1-pc) + (n01+n11)*sl(pc)
    lla = n00*sl(1-p01)+n01*sl(p01)+n10*sl(1-p11)+n11*sl(p11)
    LR = -2*(llc-lla); pv = 1 - stats.chi2.cdf(LR, 1)
    return {"LR_ind":round(LR,3),"p_value":round(pv,4),"rechaza":pv<0.05}

def christoffersen_cc(kup, ind):
    LR = kup["LR_uc"] + ind["LR_ind"]; pv = 1 - stats.chi2.cdf(LR, 2)
    return {"LR_cc":round(LR,3),"p_value":round(pv,4),"rechaza":pv<0.05}

def full_backtest(ret_oos, var_s, alpha=ALPHA):
    exc = ((-ret_oos).values > var_s.values).astype(int)
    kup = kupiec_test(exc, alpha); ind = christoffersen_independence(exc); cc = christoffersen_cc(kup, ind)
    return {"exceptions":exc,"kupiec":kup,"independence":ind,"cc":cc}

def es_exception_rate(ret_oos, es_s):
    return ((-ret_oos).values > es_s.values).astype(int).mean()

def compare_rebalanced_vs_static(ret_oos, es_s, rebal):
    mask = pd.Series(False, index=ret_oos.index)
    for d in rebal:
        if d in ret_oos.index:
            pos = ret_oos.index.get_loc(d); mask.loc[ret_oos.index[pos:pos+5]] = True
    exc = pd.Series((-ret_oos).values > es_s.values, index=ret_oos.index)
    return {"post_rebalanceo":round(exc[mask].mean(),4),"resto":round(exc[~mask].mean(),4),
            "reduce":exc[mask].mean() < exc[~mask].mean()}

def minimum_shock_for_target_loss(w, target=0.20):
    direction = w/w.sum(); denom = float(w@direction)
    km = target/denom if denom else np.inf
    uni = {TICKERS[i]: (target/w[i] if w[i] > 1e-4 else np.inf) for i in range(len(TICKERS))}
    return {"multivariado":km,"univariado":uni}

def forecast_sigma_oos(fm, horizon):
    f = fm["best_model"].forecast(horizon=horizon, method="simulation", simulations=500, reindex=False)
    return np.sqrt(f.variance.iloc[-1].values)

def plot_var_backtest(ret_oos, vdf, exc, title):
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(ret_oos.index, ret_oos.values, lw=0.6, color="#34495e", label="Retorno portafolio")
    ax.plot(vdf.index, -vdf["VaR_Normal"], color="#3498db", lw=1.0, label="-VaR Normal")
    ax.plot(vdf.index, -vdf["VaR_t"], color="#e67e22", lw=1.0, label="-VaR t")
    if "VaR_EVT" in vdf.columns:
        ax.plot(vdf.index, -vdf["VaR_EVT"], color="#c0392b", lw=1.0, label="-VaR EVT")
    ed = ret_oos.index[exc.astype(bool)]
    ax.scatter(ed, ret_oos.loc[ed], color="red", s=18, zorder=5, label="Excepciones (vs t)")
    ax.axhline(0, color="gray", lw=0.4); ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()

oos_dec = oos/100.0
w_mv = bt["weights_mv"].iloc[-1].values
w_rp = bt["weights_rp"].iloc[-1].values
last_corr = dcc_res["R_t"][-1]
print("Proyectando sigma_p en OOS con forecast() (simulacion) + ultima R_t del DCC...")
sigma_fcst = np.column_stack([forecast_sigma_oos(fitted[t], len(oos)) for t in TICKERS])/100.0
Sigma_oos = np.zeros((len(oos), len(TICKERS), len(TICKERS)))
for t in range(len(oos)):
    Dt = np.diag(sigma_fcst[t]); Sigma_oos[t] = Dt@last_corr@Dt
sp_mv = pd.Series([np.sqrt(w_mv@Sigma_oos[t]@w_mv) for t in range(len(oos))], index=oos.index)
sp_rp = pd.Series([np.sqrt(w_rp@Sigma_oos[t]@w_rp) for t in range(len(oos))], index=oos.index)
ret_mv = pd.Series(oos_dec[TICKERS].values@w_mv, index=oos.index)
ret_rp = pd.Series(oos_dec[TICKERS].values@w_rp, index=oos.index)
zin_mv = (bt["returns_mv"]-bt["returns_mv"].mean())/bt["returns_mv"].ewm(span=60).std()
zin_rp = (bt["returns_rp"]-bt["returns_rp"].mean())/bt["returns_rp"].ewm(span=60).std()

var_mv = add_evt_var(rolling_portfolio_var(ret_mv, sp_mv), zin_mv, sp_mv)
var_rp = add_evt_var(rolling_portfolio_var(ret_rp, sp_rp), zin_rp, sp_rp)
bt_mv = full_backtest(ret_mv, var_mv["VaR_t"]); bt_rp = full_backtest(ret_rp, var_rp["VaR_t"])

print("\nBACKTEST MEAN-VARIANCE (VaR t-Student)")
print("  Kupiec:", bt_mv["kupiec"]); print("  Christoffersen ind:", bt_mv["independence"])
print("  Christoffersen cc:", bt_mv["cc"])
print("\nBACKTEST RISK PARITY (VaR t-Student)")
print("  Kupiec:", bt_rp["kupiec"]); print("  Christoffersen ind:", bt_rp["independence"])
print("  Christoffersen cc:", bt_rp["cc"])

print("\nTasas de excepcion de ES (deben ser < tasa de VaR):")
for nm, rr, vv in [("MV", ret_mv, var_mv), ("RP", ret_rp, var_rp)]:
    print(f"  {nm}: ES_N={es_exception_rate(rr, vv['ES_Normal']):.4f} "
          f"ES_t={es_exception_rate(rr, vv['ES_t']):.4f} ES_EVT={es_exception_rate(rr, vv['ES_EVT']):.4f}")

rebal_oos = oos.index[::REBAL_FREQ_DAYS]
print("\nEfecto del rebalanceo sobre excepciones de ES:")
print("  MV:", compare_rebalanced_vs_static(ret_mv, var_mv["ES_t"], rebal_oos))
print("  RP:", compare_rebalanced_vs_static(ret_rp, var_rp["ES_t"], rebal_oos))

print("\nShock minimo para una perdida de portafolio del 20%:")
for nm, w in [("MV", w_mv), ("RP", w_rp)]:
    sh = minimum_shock_for_target_loss(w, 0.20)
    print(f"  {nm} multivariado (prop. a pesos): {sh['multivariado']*100:.1f}%")
    top = sorted([(t,s) for t,s in sh["univariado"].items() if np.isfinite(s)], key=lambda x: x[1])[:3]
    print(f"  {nm} univariado (activos con mayor peso): " +
          ", ".join(f"{t}: {s*100:.1f}%" for t, s in top))

plot_var_backtest(ret_mv, var_mv, bt_mv["exceptions"], "Pregunta 5 - Backtest VaR Mean-Variance (OOS)")
plot_var_backtest(ret_rp, var_rp, bt_rp["exceptions"], "Pregunta 5 - Backtest VaR Risk Parity (OOS)")